### Connection Cell

In [9]:
import duckdb
import polars as pl
import plotly.express as px

# Create an in-memory DuckDB connection
con = duckdb.connect()

# Load PostgreSQL extension and attach database
con.execute("INSTALL postgres;")
con.execute("LOAD postgres;")

# Update with your PostgreSQL credentials
pg_conn = "dbname=dvdrental user=postgres password=postgres host=localhost port=5432"
con.execute(f"ATTACH '{pg_conn}' AS pg (TYPE POSTGRES);")

print("✓ Successfully connected DuckDB to PostgreSQL!")

✓ Successfully connected DuckDB to PostgreSQL!


### Visualization Cells

In [3]:
query = """
    SELECT email 
    FROM pg.public.customer
    ORDER BY customer_id 
    LIMIT 50;
"""

df_emails: pl.DataFrame = con.execute(query).pl()
df_emails

email
str
"""mary.smith@sakilacustomer.org"""
"""patricia.johnson@sakilacustome…"
"""linda.williams@sakilacustomer.…"
"""barbara.jones@sakilacustomer.o…"
"""elizabeth.brown@sakilacustomer…"
…
"""catherine.campbell@sakilacusto…"
"""frances.parker@sakilacustomer.…"
"""ann.evans@sakilacustomer.org"""


### Revenue by Film (Top 5)

In [4]:
query_revenue = """
    SELECT f.title, SUM(p.amount) AS total_revenue
    FROM pg.public.film f
    JOIN pg.public.inventory i ON f.film_id = i.film_id
    JOIN pg.public.rental r    ON i.inventory_id = r.inventory_id
    JOIN pg.public.payment p   ON r.rental_id = p.rental_id
    GROUP BY f.title
    ORDER BY total_revenue DESC
    LIMIT 5;
"""

df_revenue = con.execute(query_revenue).pl()

# Visualize with Plotly
fig = px.bar(
    df_revenue.to_pandas(),
    x="title",
    y="total_revenue",
    title="Top 5 Revenue Generating Films",
    labels={"title": "Film Title", "total_revenue": "Total Revenue ($)"},
    text_auto=".2f",
    template="plotly_dark"
)
fig.show()

### Store Performance

In [5]:
query_store = """
    SELECT s.store_id::text AS store, COUNT(i.inventory_id) AS total_inventory
    FROM pg.public.store s
    JOIN pg.public.inventory i ON s.store_id = i.store_id
    GROUP BY s.store_id;
"""

df_store = con.execute(query_store).pl()

# Visualize with Plotly
fig_store = px.pie(
    df_store.to_pandas(),
    names="store",
    values="total_inventory",
    title="Inventory Count by Store",
    hole=0.4,
    template="plotly_dark"
)
fig_store.show()

### Popular Categories

In [6]:
query_categories = """
    SELECT c.name AS category_name, COUNT(fc.film_id) AS total_films
    FROM pg.public.category c
    JOIN pg.public.film_category fc ON c.category_id = fc.category_id
    GROUP BY c.name
    HAVING COUNT(fc.film_id) > 60
    ORDER BY total_films DESC;
"""

df_cat = con.execute(query_categories).pl()

# Visualize with Plotly
fig_cat = px.bar(
    df_cat.to_pandas(),
    x="total_films",
    y="category_name",
    orientation="h",
    title="Categories with More Than 60 Films",
    labels={"category_name": "Category", "total_films": "Total Films"},
    template="plotly_dark"
)
fig_cat.show()

### Customers living in London

In [7]:
query_london = """
    SELECT c.first_name, c.last_name, c.email, ci.city
    FROM pg.public.customer c
    JOIN pg.public.address a ON c.address_id = a.address_id
    JOIN pg.public.city ci   ON a.city_id = ci.city_id
    WHERE ci.city = 'London';
"""

df_london = con.execute(query_london).pl()
df_london

first_name,last_name,email,city
str,str,str,str
"""Mattie""","""Hoffman""","""mattie.hoffman@sakilacustomer.…","""London"""
"""Cecil""","""Vines""","""cecil.vines@sakilacustomer.org""","""London"""
